# 14 - Ensemble (WER-Weighted Voting) Inference

**Technique:** WER-weighted character-level voting  
**Approach:** Weight each model's vote by inverse WER (better models get more weight)

**Models & Blind Test Scores:**
| Model | WER | Weight (1/WER) |
|-------|-----|----------------|
| seamless_m4t | 0.10 | 1.0 |
| arabert | 0.10 | 1.0 |
| whisper_quran_lora | 0.10 | 1.0 |
| catt | 0.10 | 1.0 |
| tarteel_whisper | 0.10 | 1.0 |
| whisper_multimodal | 0.11 | 0.91 |

**Tasks:**
1. Dev Set: Load predictions + Weighted Ensemble + Metrics
2. Blind Test: Load predictions + Weighted Ensemble + Submission JSON

In [1]:
!pip install -q tqdm


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Setup - Import from config.py (generated by setup.sh)
import os, sys, json, re, zipfile
from datetime import datetime
from collections import Counter, defaultdict
from pathlib import Path
from tqdm import tqdm

# Add project root to path for config import
IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))

# Import paths from config
try:
    from config import (
        PROJECT_DIR, DATA_DIR,
        DEV_OUTPUT,
        OUTPUT_DIR, SUBMISSION_DIR
    )
except ImportError:
    print("WARNING: config.py not found. Run setup.sh first!")
    DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
    DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
    OUTPUT_DIR = PROJECT_DIR / 'outputs'
    SUBMISSION_DIR = PROJECT_DIR / 'submissions'

MODEL_KEY = 'ensemble_weighted'

# Weights based on blind test WER (inverse WER, normalized)
# Lower WER = higher weight
MODEL_WEIGHTS = {
    'seamless_m4t':       1.0,   # WER: 0.10
    'arabert':            1.0,   # WER: 0.10
    'whisper_quran_lora': 1.0,   # WER: 0.10
    'catt':               1.0,   # WER: 0.10
    'tarteel_whisper':    1.0,   # WER: 0.10
    'whisper_multimodal': 0.91,  # WER: 0.11
}

print(f"Environment: 'Local'")
print(f"Model weights ({len(MODEL_WEIGHTS)} models):")
for model, weight in MODEL_WEIGHTS.items():
    print(f"  {model}: {weight:.2f}")

Environment: Local
Model weights (6 models):
  seamless_m4t: 1.00
  arabert: 1.00
  whisper_quran_lora: 1.00
  catt: 1.00
  tarteel_whisper: 1.00
  whisper_multimodal: 0.91


In [3]:
# Load predictions
def load_predictions(model_keys, pred_type='dev'):
    all_preds = {}
    for model_key in model_keys:
        pred_file = OUTPUT_DIR / f"{model_key}_{pred_type}_predictions.json"
        if pred_file.exists():
            with open(pred_file, 'r', encoding='utf-8') as f:
                preds = json.load(f)
                all_preds[model_key] = {p['id']: p['text_diacritized'] for p in preds}
            print(f"Loaded {model_key}: {len(all_preds[model_key])} samples")
    return all_preds

In [4]:
# Diacritic patterns
DIACRITIC_PATTERN = re.compile(r'[\u064B-\u0652]')
ARABIC_LETTER_PATTERN = re.compile(r'[\u0621-\u064A]')
IRAB_DIACRITICS = {'\u064B', '\u064C', '\u064D'}

def extract_char_diacritics(text):
    result = []
    i = 0
    while i < len(text):
        char = text[i]
        if ARABIC_LETTER_PATTERN.match(char):
            diacritics = []
            j = i + 1
            while j < len(text) and DIACRITIC_PATTERN.match(text[j]):
                diacritics.append(text[j])
                j += 1
            result.append((char, ''.join(diacritics)))
            i = j
        else:
            result.append((char, ''))
            i += 1
    return result

def reconstruct_text(char_diac_pairs):
    return ''.join(char + diac for char, diac in char_diac_pairs)

In [5]:
def weighted_ensemble_vote(texts_with_weights):
    """
    Weighted voting ensemble at character level.
    
    Args:
        texts_with_weights: List of (text, weight) tuples
    """
    if not texts_with_weights:
        return ""
    
    if len(texts_with_weights) == 1:
        return texts_with_weights[0][0]
    
    # Extract char+diacritic pairs with weights
    pairs_with_weights = [(extract_char_diacritics(t), w) for t, w in texts_with_weights]
    
    # Find most common length
    lengths = Counter(len(p) for p, w in pairs_with_weights)
    target_len = lengths.most_common(1)[0][0]
    
    # Filter to texts with target length
    valid_pairs = [(p, w) for p, w in pairs_with_weights if len(p) == target_len]
    
    if not valid_pairs:
        return texts_with_weights[0][0]
    
    # Weighted vote at each position
    result_pairs = []
    for pos in range(target_len):
        # Accumulate weighted votes for each (char, diac)
        char_votes = defaultdict(float)
        diac_votes = defaultdict(float)
        
        for pairs, weight in valid_pairs:
            char, diac = pairs[pos]
            char_votes[char] += weight
            diac_votes[diac] += weight
        
        # Pick highest weighted
        voted_char = max(char_votes, key=char_votes.get)
        voted_diac = max(diac_votes, key=diac_votes.get)
        
        result_pairs.append((voted_char, voted_diac))
    
    return reconstruct_text(result_pairs)

In [6]:
def create_weighted_ensemble(all_preds, model_weights, sample_ids):
    results = []
    
    for sample_id in tqdm(sample_ids, desc="Weighted ensemble"):
        texts_with_weights = []
        
        for model_key, weight in model_weights.items():
            if model_key in all_preds and sample_id in all_preds[model_key]:
                texts_with_weights.append((all_preds[model_key][sample_id], weight))
        
        if texts_with_weights:
            ensemble_text = weighted_ensemble_vote(texts_with_weights)
        else:
            ensemble_text = ""
        
        results.append({'id': sample_id, 'text_diacritized': ensemble_text})
    
    return results

In [7]:
# Metrics
def compute_metrics(predictions, ground_truth, exclude_irab=False):
    gt_lookup = {item['id']: item['text_diacritized'] for item in ground_truth}
    total_chars, total_words, char_errors, word_errors, ser_sum, n = 0, 0, 0, 0, 0, 0
    
    for pred in predictions:
        if pred['id'] not in gt_lookup: continue
        pred_text, ref_text = pred['text_diacritized'], gt_lookup[pred['id']]
        pred_pairs = [(c, d) for c, d in extract_char_diacritics(pred_text) if ARABIC_LETTER_PATTERN.match(c)]
        ref_pairs = [(c, d) for c, d in extract_char_diacritics(ref_text) if ARABIC_LETTER_PATTERN.match(c)]
        
        for i in range(min(len(pred_pairs), len(ref_pairs))):
            pred_d, ref_d = pred_pairs[i][1], ref_pairs[i][1]
            if exclude_irab:
                pred_d = ''.join(d for d in pred_d if d not in IRAB_DIACRITICS)
                ref_d = ''.join(d for d in ref_d if d not in IRAB_DIACRITICS)
            if pred_d != ref_d: char_errors += 1
        char_errors += abs(len(pred_pairs) - len(ref_pairs))
        total_chars += max(len(pred_pairs), len(ref_pairs))
        
        pred_words, ref_words = pred_text.split(), ref_text.split()
        for i in range(min(len(pred_words), len(ref_words))):
            if pred_words[i] != ref_words[i]: word_errors += 1
        word_errors += abs(len(pred_words) - len(ref_words))
        total_words += max(len(pred_words), len(ref_words))
        if pred_text != ref_text: ser_sum += 1
        n += 1
    
    return {'DER': char_errors/total_chars if total_chars else 0, 'WER': word_errors/total_words if total_words else 0, 'SER': ser_sum/n if n else 0, 'n_samples': n}

## Dev Set Ensemble

In [8]:
dev_preds = load_predictions(list(MODEL_WEIGHTS.keys()), 'dev')

all_dev_ids = set()
for model_preds in dev_preds.values():
    all_dev_ids.update(model_preds.keys())

print(f"\nTotal dev samples: {len(all_dev_ids)}")

Loaded seamless_m4t: 260 samples
Loaded arabert: 260 samples
Loaded whisper_quran_lora: 260 samples
Loaded catt: 260 samples
Loaded tarteel_whisper: 260 samples
Loaded whisper_multimodal: 260 samples

Total dev samples: 260


In [9]:
dev_ensemble = create_weighted_ensemble(dev_preds, MODEL_WEIGHTS, sorted(all_dev_ids))

with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    dev_output = json.load(f)

print("="*60 + "\nDEV SET RESULTS (WEIGHTED ENSEMBLE)\n" + "="*60)
m1 = compute_metrics(dev_ensemble, dev_output, False)
print(f"\n[With I'rab] DER: {m1['DER']*100:.2f}% | WER: {m1['WER']*100:.2f}% (PRIMARY) | SER: {m1['SER']*100:.2f}%")
m2 = compute_metrics(dev_ensemble, dev_output, True)
print(f"[No I'rab]   DER: {m2['DER']*100:.2f}% | WER: {m2['WER']*100:.2f}% | SER: {m2['SER']*100:.2f}%")

with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_ensemble, f, ensure_ascii=False, indent=2)

Weighted ensemble: 100%|██████████| 260/260 [00:00<00:00, 5087.47it/s]

DEV SET RESULTS (WEIGHTED ENSEMBLE)

[With I'rab] DER: 13.18% | WER: 40.87% (PRIMARY) | SER: 78.46%
[No I'rab]   DER: 13.13% | WER: 40.87% | SER: 78.46%


## Blind Test Ensemble

In [10]:
test_preds = load_predictions(list(MODEL_WEIGHTS.keys()), 'test')

all_test_ids = set()
for model_preds in test_preds.values():
    all_test_ids.update(model_preds.keys())

print(f"\nTotal test samples: {len(all_test_ids)}")

Loaded seamless_m4t: 328 samples
Loaded arabert: 328 samples
Loaded whisper_quran_lora: 328 samples
Loaded catt: 328 samples
Loaded tarteel_whisper: 328 samples
Loaded whisper_multimodal: 328 samples

Total test samples: 328


In [11]:
test_ensemble = create_weighted_ensemble(test_preds, MODEL_WEIGHTS, sorted(all_test_ids))

with open(OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(test_ensemble, f, ensure_ascii=False, indent=2)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_ensemble, f, ensure_ascii=False, indent=2)
zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print(f"\nSUBMISSION READY: {zip_file} ({zip_file.stat().st_size/1024:.1f} KB)")

Weighted ensemble: 100%|██████████| 328/328 [00:00<00:00, 5834.61it/s]


SUBMISSION READY: ./submissions/ensemble_weighted_submission_20260220_152533.zip (18.4 KB)
